# SAC Irrigation Training - v2.15 (reward-shape recovery via linear r6, Kaggle)

**Algorithm:** SAC (stable_baselines3) with VDN-factorised twin-Q + LayerNorm critic (byte-identical to v2.11)
**Actor:** LeakyReLU(0.01) MLP, trained on NORMALISED global/forecast observations
**Learning rate:** asymmetric - actor LR = 5x critic LR
**Entropy:** alpha = 0.01 (inherited from v2.14)
**Reward r6:** LINEAR in overshoot (v2.15 single-variable change vs v2.14)

## Why v2.15 exists

v2.14 (LayerNorm critic + LeakyReLU actor + actor input re-center + alpha=0.01)
achieved mean yield 3710 kg/ha on the 9-cell perfect-forecast grid (matching
v2.7's transient cascade-peak with stable training).  But behavioural diagnostics
on the wet/100 rollout exposed a concrete defect:

| metric                  | v2.14   | MPC     | gap                              |
|-------------------------|---------|---------|----------------------------------|
| daily u 10th-pctile     | 2.50    | 0.16    | cannot push action low           |
| daily u 1st-pctile      | 2.65    | 0.03    | hard ~2.5 mm floor               |
| corr(u, rain_fwd7)      | +0.03   | -0.42   | FORECAST blindness               |
| waterlog days/agent     | 80.2    | 19.0    | 4x more saturated days           |
| WUE (kg/ha/mm)          | 8.62    | 12.12   | 12% efficiency gap               |

The actor IS state-responsive (correct-sign corr with x1, x5, ET, today's rain)
but cannot reduce u toward zero on rainy days.  Two compounding causes both
addressed by linear r6:

1. **Quadratic r6 has weak marginal penalty at moderate overshoot.**
   `d(r6_quad)/d(overshoot) = -2*ALPHA6*overshoot/FC^2`.  At overshoot=12 mm
   (typical wet operating point) the gradient is ~0.01; at overshoot=2 mm
   (dry-year transients) it is essentially zero.  The critic's dQ/du signal
   at the policy's operating point is small.

2. **The ABM's waterlog stress h6 is LINEAR in (x1-FC)/FC.**  Quadratic r6
   is a *misaligned proxy* for the physical yield loss.  Linear r6 aligns
   the reward shape with the simulator's actual physics term.

## The v2.15 change (one variable)

`r6 = -ALPHA6_LIN * mean(overshoot) / FC`,  ALPHA6_LIN = **1.5**

ALPHA6_LIN calibrated from 27 v2.14+MPC rollouts:
- match full-distribution season-sum: ALPHA6_LIN ~1.47
- match wet-only season-sum:           ALPHA6_LIN ~1.69
- match |dQ/du| at RMS overshoot 13.4mm: ALPHA6_LIN ~1.53

Architecturally byte-identical to v2.14 / v2.13.  All other hyperparameters
unchanged: V211 LayerNorm critic, LeakyReLU actor, actor input re-center,
normalised globals, asymmetric actor LR (5x), gamma=0.99, tau=0.005,
alpha=0.01, batch 256, buffer 250k, 250k steps, MAX_GRAD_NORM=1.0.


In [ ]:
# Cell 1: Clone repo and install deps.
import subprocess, sys, os

WORK = '/kaggle/working'
repo = os.path.join(WORK, 'thesis')
if os.path.exists(repo):
    subprocess.run(['rm', '-rf', repo], check=True)
subprocess.run(
    ['git', 'clone', 'https://github.com/taratorbati/thesis.git', repo],
    check=True)

os.chdir(repo)
sys.path.insert(0, repo)

subprocess.run(
    ['pip', 'install', '--quiet',
     'stable-baselines3==2.6.0', 'gymnasium', 'wandb', 'pytest'],
    check=True)

import torch
print(f'PyTorch:        {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU:            {torch.cuda.get_device_name(0)}')


In [ ]:
# Cell 2: WandB secret + GPU check.
import os
try:
    from kaggle_secrets import UserSecretsClient
    os.environ['WANDB_API_KEY'] = UserSecretsClient().get_secret('WANDB_API_KEY')
    print('OK  WANDB_API_KEY loaded from Kaggle Secrets.')
except Exception as e:
    print(f'NOTE: Could not load WANDB_API_KEY ({type(e).__name__}).')
    print('     Training continues without WandB - add it via Add-ons > Secrets to enable it.')

import subprocess
r = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(r.stdout if r.returncode == 0 else 'nvidia-smi failed - no GPU allocated')


In [ ]:
# Cell 3: Pre-training validation.
#
# Runs smoke tests, factorized-critic tests (now including v2.15 marker guard
# and reward_overshoot_mode validation), and a 1000-step pilot to catch
# import/wiring bugs.  Abort if anything fails.
import subprocess, sys

print('Smoke tests...')
r = subprocess.run(
    [sys.executable, '-m', 'pytest', 'tests/test_rl_smoke.py', '-v', '--tb=short'],
    capture_output=False)
assert r.returncode == 0, 'SMOKE TESTS FAILED'

print('\nFactorized-critic tests (v2.7 + v2.11 + v2.12 + v2.13 + v2.14 + v2.15)...')
r = subprocess.run(
    [sys.executable, '-m', 'pytest', 'tests/test_factorized_critic.py', '-v', '--tb=short'],
    capture_output=False)
assert r.returncode == 0, 'FACTORIZED CRITIC TESTS FAILED'

print('\n1000-step pilot training (wiring check, ~1-2 min)...')
from src.rl.train_v215 import train_sac_v215
_ = train_sac_v215(
    seed=999,
    output_dir='/kaggle/working/pilot',
    wandb_project=None,
    total_timesteps=1000,
)
print('\nOK  Pre-flight passed. Proceed to Cell 4.')


In [ ]:
# Cell 4: Full 250k training (SAC v2.15 - linear r6, alpha=0.01, gamma=0.99).
# ~30-55 min on A100, ~2-2.5 h on T4.
#
# Start with SEED=0 (paired with v2.7 / v2.11 / v2.14 seed 0). Expand to seeds 1, 2
# only after seed-0 meets the primary acceptance criteria (action floor drops,
# wet/100 forecast correlation goes negative).

SEED = 0       # CHANGE per session

from src.rl.train_v215 import train_sac_v215

model = train_sac_v215(
    seed=SEED,
    output_dir='/kaggle/working/thesis/results/rl',
    wandb_project='sac-irrigation-thesis',
    total_timesteps=250_000,
    gamma=0.99,
    actor_lr_mult=5.0,                  # actor LR = 5x critic LR
    ent_coef=0.01,                      # inherited from v2.14
    reward_overshoot_mode='linear',     # *** THE v2.15 CHANGE ***
)
print('Training complete.')


In [ ]:
# Cell 5: Archive results so Kaggle persists them after the session.
import shutil, os, datetime

src = f'/kaggle/working/thesis/results/rl/sac_v215_seed{SEED}'
timestamp = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
dst = f'/kaggle/working/sac_v215_seed{SEED}_{timestamp}'

shutil.copytree(src, dst, ignore=shutil.ignore_patterns('replay_buffer_latest.pkl'))
print(f'Archived to: {dst}  (download from the Kaggle output panel)')
for root, _, files in os.walk(dst):
    for f in files:
        p = os.path.join(root, f); size = os.path.getsize(p)
        print(f'  {os.path.relpath(p, dst)}  ({size/1024:.1f} KB)')


In [ ]:
# Cell 6: Post-training 9-cell evaluation (SAC eval path).
#
# v2.15 produces a SAC checkpoint with marker=2.15. The runner auto-detects it
# via the 'actor.obs_norm_marker' buffer (+ the 1-D 'critic.qf0.1.weight'
# LayerNorm key) and dispatches to V215CTDESACPolicy.  Eval uses the default
# (quadratic) r6 in the env - this does NOT affect yield/water/waterlog metrics
# (those are physics-driven from the ABM), only the eval-reward print value.
import subprocess, sys

# Use the LAST checkpoint by default - the diagnostic in Cell 7 may suggest
# preferring a different one based on responsiveness vs yield trade-off.
model_path = f'/kaggle/working/thesis/results/rl/sac_v215_seed{SEED}/best_model/best_model.zip'

print('Evaluating on 9-cell grid (perfect forecast)...')
r = subprocess.run([
    sys.executable, '-m', 'scripts.experiments.exp_rl',
    '--mode',     'eval',
    '--model',    model_path,
    '--scenario', 'all',
    '--budget',   'all',
    '--forecast', 'perfect',
], capture_output=False)
assert r.returncode == 0, 'PERFECT-FORECAST EVAL FAILED'


In [ ]:
# Cell 7: Actor diagnostic - "did the action floor drop?"
#
# This is the v2.15 ACCEPTANCE check.  v2.14's defect was that the wet-year
# daily action 10th-percentile was 2.50 mm (vs MPC's 0.16 mm).  If linear r6
# works, this number should drop below ~1.5 mm.  We also check the
# forecast-response correlation, which was +0.03 in v2.14 (essentially blind)
# vs -0.42 in MPC.
import numpy as np, torch, glob, os, re
import matplotlib.pyplot as plt
import torch.nn as nn
from src.rl.gym_env import IrrigationEnv

ckpt_dir = f'/kaggle/working/thesis/results/rl/sac_v215_seed{SEED}/checkpoints'
paths = sorted(glob.glob(os.path.join(ckpt_dir, f'sac_v215_seed{SEED}_*_steps.zip')),
               key=lambda p: int(re.search(r'_(\d+)_steps', p).group(1)))

def load_actor(zip_path):
    import zipfile, io
    with zipfile.ZipFile(zip_path) as zf:
        sd = torch.load(io.BytesIO(zf.read('policy.pth')), map_location='cpu', weights_only=False)
    a = nn.ModuleDict({
        'l0': nn.Linear(65,128), 'l2': nn.Linear(128,128), 'mu': nn.Linear(128,1)})
    a['l0'].weight.data = sd['actor.latent_pi.0.weight']; a['l0'].bias.data = sd['actor.latent_pi.0.bias']
    a['l2'].weight.data = sd['actor.latent_pi.2.weight']; a['l2'].bias.data = sd['actor.latent_pi.2.bias']
    a['mu'].weight.data = sd['actor.mu.weight'];          a['mu'].bias.data = sd['actor.mu.bias']
    return a, sd

def o2p(obs, N=130, F=8):
    per = obs[:F*N].reshape(N,F); g = obs[F*N:]
    per_re = 2*per - 1.0   # v2.13/v2.14/v2.15 input re-center
    g_re   = 2*g   - 1.0
    return np.concatenate([per_re, np.broadcast_to(g_re,(N,g_re.shape[0]))], axis=1).astype(np.float32)

# Use a fixed wet/100 scenario for the wet-year diagnostic (the case that
# matters).  IrrigationEnv with randomize=False picks the default scenario;
# we will manually invoke the wet-year settings via the same scripts the
# 9-cell eval uses.  For a quick checkpoint sweep we use a single deterministic
# rollout per checkpoint.
env = IrrigationEnv(randomize=False, curriculum_warmup_steps=0,
                    use_overshoot_feature=False, normalize_globals=True,
                    reward_overshoot_mode='linear')   # match training-time

steps, alive_l, mutstd_l, muwstd_l, u_p10_l, u_min_l, u_max_l = [], [], [], [], [], [], []
for p in paths:
    actor, sd = load_actor(p)
    obs,_ = env.reset(); alive=[]; mus=[]; us=[]
    for d in range(93):
        pa = torch.from_numpy(o2p(obs))
        h1 = torch.nn.functional.leaky_relu(actor['l0'](pa), 0.01)
        h2 = torch.nn.functional.leaky_relu(actor['l2'](h1), 0.01)
        mu = actor['mu'](h2)
        alive.append((h1>0).any(dim=0).sum().item())
        mus.append(mu.mean().item())
        a = (np.tanh(mu.detach().numpy().flatten())*0.5+0.5)
        us.append(float((a*12.0).mean()))
        obs,_,term,trunc,_ = env.step(a)
        if term or trunc: obs,_ = env.reset()
    us = np.array(us)
    step = int(re.search(r'_(\d+)_steps', p).group(1))
    steps.append(step); alive_l.append(np.mean(alive))
    mutstd_l.append(np.std(mus)); muwstd_l.append(sd['actor.mu.weight'].std().item())
    u_p10_l.append(float(np.percentile(us,10)))
    u_min_l.append(float(us.min()))
    u_max_l.append(float(us.max()))
    print(f'step {step:>7}: live~{np.mean(alive):5.1f}/128  mu_std={np.std(mus):.4f}  '
          f'u in [{us.min():.2f}, {us.max():.2f}]  u_p10={np.percentile(us,10):.2f}  '
          f'mu.w_std={sd["actor.mu.weight"].std().item():.4f}')

fig, ax = plt.subplots(1, 4, figsize=(18, 4))
ax[0].plot(steps, alive_l, '-o'); ax[0].axhline(30, color='r', ls=':')
ax[0].set_title('live first-layer units / 128'); ax[0].set_xlabel('step'); ax[0].grid(alpha=.3)
ax[1].plot(steps, mutstd_l, '-o'); ax[1].axhline(0.20, color='r', ls=':')
ax[1].set_title('pre-tanh mu temporal std (state response)'); ax[1].set_xlabel('step'); ax[1].grid(alpha=.3)
ax[2].plot(steps, u_p10_l, '-o', label='u 10th-pctile')
ax[2].plot(steps, u_min_l, '-^', label='u daily min')
ax[2].axhline(1.5, color='r', ls=':', label='v2.15 target u_p10 < 1.5')
ax[2].axhline(2.50, color='gray', ls='--', label='v2.14 u_p10 = 2.50')
ax[2].set_title('action floor (mm/day)'); ax[2].set_xlabel('step'); ax[2].legend(); ax[2].grid(alpha=.3)
ax[3].plot(steps, muwstd_l, '-o'); ax[3].axhline(0.10, color='r', ls=':')
ax[3].set_title('mu.weight std (actor learning)'); ax[3].set_xlabel('step'); ax[3].grid(alpha=.3)
plt.tight_layout(); plt.show()
print('\nv2.15 PASS if: u_p10 drops below ~1.5 mm (was 2.50 in v2.14, MPC = 0.16).')
print('              live units stay >= 30, mu_std stays >= 0.20.')
print('Compare against v2.14: u_p10=2.50, u_min=2.65, mu_std=0.24 at 250k.')


In [ ]:
# Cell 8: v2.14 vs v2.15 behavioural comparison on the 9-cell grid.
#
# Loads the just-finished v2.15 eval JSONs and prints the head-to-head against
# v2.14's results.  This is the THESIS-RELEVANT comparison: did linear r6
# close the wet-year over-irrigation gap?
import json, glob, os, numpy as np

ORDER = [('dry','100pct'),('dry','85pct'),('dry','70pct'),
         ('moderate','100pct'),('moderate','85pct'),('moderate','70pct'),
         ('wet','100pct'),('wet','85pct'),('wet','70pct')]

def grab(pattern):
    res = {}
    for f in sorted(glob.glob(pattern)):
        if 'seed1' in os.path.basename(f): continue
        j = json.load(open(f)); fn = os.path.basename(f)
        pct = [p for p in ['100pct','85pct','70pct'] if p in fn][0]
        m = j['final_metrics']
        res[(j['scenario'],pct)] = (m['yield_kg_ha'], m.get('water_used_mm'),
                                     m.get('waterlog_days_per_agent'),
                                     m.get('wue_kg_ha_per_mm'))
    return res

# v2.15 (just trained):
v215_dir = f'/kaggle/working/thesis/results/runs/sac_v215_best_model'
# fall back to relative path under the eval output directory if needed
if not os.path.exists(v215_dir):
    v215_dir = 'results/runs/sac_v215_best_model'
v215 = grab(os.path.join(v215_dir, 'sac_perfect_det_*seed0.json'))

# v2.14 (baseline - committed in repo):
v214 = grab('results/runs/sac_v214_best_model/sac_perfect_det_*seed0.json')

# MPC oracle:
mpc = grab('results/runs/mpc_perfect_*_Hp8.json')

def mean(d, i):
    vs = [d[o][i] for o in ORDER if o in d and d[o][i] is not None]
    return float(np.mean(vs)) if vs else float('nan')

print('='*100)
print(' v2.15 vs v2.14 vs MPC  -  9-cell perfect-forecast grid, seed 0')
print('='*100)
print(f'%-20s %18s %18s %18s' % ('scenario','v2.15 (y/mm/wlog)','v2.14 (y/mm/wlog)','MPC (y/mm/wlog)'))
for o in ORDER:
    def fmt(d):
        if o not in d: return 'NA'
        y,w,wl,_ = d[o]; return f'{y:.0f}/{w or 0:.0f}/{wl or 0:.0f}'
    print(f'%-20s %18s %18s %18s' % (o[0]+'/'+o[1], fmt(v215), fmt(v214), fmt(mpc)))
print('-'*100)
print(f'MEAN yield      v2.15={mean(v215,0):7.1f}  v2.14={mean(v214,0):7.1f}  MPC={mean(mpc,0):7.1f}')
print(f'MEAN water      v2.15={mean(v215,1):7.1f}  v2.14={mean(v214,1):7.1f}  MPC={mean(mpc,1):7.1f}')
print(f'MEAN waterlog   v2.15={mean(v215,2):7.2f}  v2.14={mean(v214,2):7.2f}  MPC={mean(mpc,2):7.2f}')
print(f'MEAN WUE        v2.15={mean(v215,3):7.2f}  v2.14={mean(v214,3):7.2f}  MPC={mean(mpc,3):7.2f}')
print()
print('Acceptance:')
print(f'  - Wet-year yield rises:        v2.15 wet_mean = '
      f"{np.mean([v215[o][0] for o in ORDER if o[0]=='wet' and o in v215]):.0f} kg/ha "
      f"(v2.14 = 3444; target >= 3500)")
print(f'  - Dry-year yield stays high:   v2.15 dry_mean = '
      f"{np.mean([v215[o][0] for o in ORDER if o[0]=='dry' and o in v215]):.0f} kg/ha "
      f"(v2.14 = 3975; floor 3900)")
print(f'  - Water use drops in wet:      v2.15 wet_water_mean = '
      f"{np.mean([v215[o][1] for o in ORDER if o[0]=='wet' and o in v215]):.0f} mm "
      f"(v2.14 = 408; MPC = 308; target <= 380)")


In [ ]:
# Cell 9: Resume from a saved checkpoint (if the session was interrupted).
# Upload the prior output as a dataset, then fill in the path.

# SEED = 0
# CHECKPOINT_STEP = 100_000
# CHECKPOINT_PATH = f'/kaggle/input/<your-dataset>/sac_v215_seed{SEED}/checkpoints/sac_v215_seed{SEED}_{CHECKPOINT_STEP}_steps.zip'
#
# from src.rl.train_v212 import AsymmetricLRSAC
# from src.rl.networks import V215CTDESACPolicy
# model = AsymmetricLRSAC.load(CHECKPOINT_PATH, custom_objects={'policy_class': V215CTDESACPolicy})
# # Continue: model.learn(total_timesteps=..., reset_num_timesteps=False)
